# Initialize: CAGE-2 network defense
Defines the reduced CAGE-2 interface, the latent context, the defender's two-part representation, the stage-game solver, the networks, the training updates, the baseline architectures, and the defender wrappers used by the other notebooks. It requires CybORG from the CAGE Challenge 2 repository (see the README).

Conventions:
- **Reduced interface.** Over the user hosts User1 to User4 and the enterprise servers Enterprise0 and Enterprise1, blue (the defender, Player 1) has $|A^1| = 15$ actions: Sleep; Monitor, in four groups; Analyse and Restore on each enterprise host; and Remove on every host. Red (the attacker, Player 2) has $|A^2| = 17$ actions over a foothold phase on the user hosts and an enterprise phase that discovers, exploits, escalates, and Impacts its targets. A red action that red cannot take yet becomes Sleep.
- **Round order.** Each round red acts first, then blue, and the true state is read from the simulator after both moves.
- **Latent context** $\theta = (p_0, p_1, w_0, w_1, h)$, drawn once per episode by `Sample_Context`:
  - red's normalized target weights $w$ over the two enterprise hosts, and a head start of $h \in \{0, \dots, 5\}$ red-only steps before the episode, both known only to red;
  - an Impact disruption probability $p_i \in \{0, 0.5, 1\}$ per enterprise host, unknown to both players. Impact on host $i$ fails with probability $p_i$, and at least one targeted host has $p_i < 1$.
- **State and observations.** The true state $x_k$ holds each host's red session level (none, user, root) and a ransomware flag per enterprise host (`System_Real_State`). Blue never sees it; Monitor and Analyse return one partial signal per round, and Remove, Restore, and Sleep return none (`extract_y`).
- **Payoffs.** Training uses the defender's shaped payoff `Reward`: host and server protection, plus bonuses for observations that change blue's picture and penalties for idle or redundant actions. Evaluation uses the physical payoff `Reward2`, protection only. The critic learns the defender's payoff, and blue is the row maximizer in the stage game; this is the paper's convention with the sign flipped.
- **Representation** $z_k = [z^S_k, z^\Theta_k]$:
  - the state component $z^S$ is blue's tracked picture of session levels and ransomware (`blue_state_vector`);
  - the context component $z^\Theta$ is a Transformer over the history rows $[z^S_{k-1}, a^1_{k-1}, z^S_k]$ (`TSTransformer`).
- **Red's private state** $m^2_k$: what red knows of its own progress (`red_state_vector`).
- **Encoder objective**, Eq. (3): $\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$ (`Train_Encoder`).
- **Game step.** The critic is trained by TD regression onto the minimax value of the next stage game, and both actors are distilled toward its equilibrium strategies by cross-entropy. Embeddings are recomputed with the current encoder at every update.

In [ ]:
import os
import copy
import json
import inspect
import random
import ipaddress
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import linprog
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.distributions import Categorical

from CybORG import CybORG
from CybORG.Shared.Actions.Action import Sleep
from CybORG.Shared.Actions import (DiscoverRemoteSystems, DiscoverNetworkServices, ExploitRemoteService,
                                   PrivilegeEscalate, Impact, Monitor, Analyse, Restore, Remove)
from CybORG.Shared.Enums import TrinaryEnum

torch.backends.mha.set_fastpath_enabled(False)      # keeps the padding-mask path identical in train and eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Reduced interface and the simulator

In [ ]:
Blue_Action_Space = ['Sleep',
                     'Monitor_User1_2', 'Monitor_User3_4', 'Monitor_Enterprise0', 'Monitor_Enterprise1',
                     'Analyse_Enterprise0', 'Analyse_Enterprise1',
                     'Remove_User1', 'Remove_User2', 'Remove_User3', 'Remove_User4', 'Remove_Enterprise0', 'Remove_Enterprise1',
                     'Restore_Enterprise0', 'Restore_Enterprise1']
Red_Action_Space = ['Sleep',
                    'DiscoverNetworkServices_Enterprise0', 'DiscoverNetworkServices_Enterprise1',
                    'ExploitRemoteService_User1', 'ExploitRemoteService_User2', 'ExploitRemoteService_User3', 'ExploitRemoteService_User4',
                    'ExploitRemoteService_Enterprise0', 'ExploitRemoteService_Enterprise1',
                    'PrivilegeEscalate_User1', 'PrivilegeEscalate_User2', 'PrivilegeEscalate_User3', 'PrivilegeEscalate_User4',
                    'PrivilegeEscalate_Enterprise0', 'PrivilegeEscalate_Enterprise1',
                    'Impact_Enterprise0', 'Impact_Enterprise1']
n_a = len(Blue_Action_Space)            # 15
n_u = len(Red_Action_Space)             # 17
real_state_dim = 8                      # session level of 6 hosts, ransomware flag of 2 enterprise hosts
blue_state_dim = 11                     # z^S
red_state_dim = 12                      # m^2
Len_Theta = 5                           # theta = [p_0, p_1, w_0, w_1, h]
Embedding_Size = 64
max_steps = 50                          # rounds per episode, also the context window
CTX_DIM = blue_state_dim * 2 + n_a      # history row: [z_s, action one-hot, next z_s]


class MyBlueAgent:
    # Placeholder agents: every action is supplied through cyborg.step
    def __init__(self): self.action_space = None
    def set_initial_values(self, action_space, observation): self.action_space = action_space
    def get_action(self, observation, action_space): return Sleep()
    def train(self, results): pass
    def end_episode(self): pass
    def get_action_space(self): return self.action_space

class MyRedAgent(MyBlueAgent):
    pass


SCENARIO_PATH = os.path.join(os.path.dirname(inspect.getfile(CybORG)), 'Shared', 'Scenarios', 'Scenario1b.yaml')
cyborg = CybORG(SCENARIO_PATH, 'sim', agents={'Blue': MyBlueAgent, 'Red': MyRedAgent})

## Red side: knowledge, private state, actions, scripted policy

In [ ]:
def red_knowledge_dic(observation, red_knowledge):
    if 'User0' in observation:
        host_info = observation['User0']
        ip = None
        subnet = None
        for iface in host_info.get('Interface', []):
            if ip is None and 'IP Address' in iface:
                ip = str(iface['IP Address'])
            if subnet is None and 'Subnet' in iface:
                subnet = str(iface['Subnet'])
        for sess in host_info.get('Sessions', []):
            if sess.get('Agent') == 'Red':
                red_knowledge['User0'] = {'ip': ip, 'subnet': subnet, 'session': sess['ID']}

    ip_to_hostname = {}
    for key, val in observation.items():
        if key.count('.') != 3:
            continue
        sysinfo = val.get('System info', {})
        if 'Hostname' in sysinfo:
            ip_to_hostname[key] = sysinfo['Hostname']

    known_ips = {info.get('ip') for info in red_knowledge.values()}
    new_ips = sorted([k for k in observation if k.count('.') == 3 and k not in known_ips])[:4]
    for i, ip in enumerate(new_ips):
        label = f'User{i+1}'
        if label not in red_knowledge:
            red_knowledge[label] = {'ip': ip}

    for ip_key, val in observation.items():
        if ip_key.count('.') != 3:
            continue
        sysinfo = val.get('System info', {})
        hostname = sysinfo.get('Hostname')
        if hostname and hostname in ['User1', 'User2', 'User3', 'User4']:
            for host, info in red_knowledge.items():
                if info.get('ip') == ip_key and 'hostname' not in info:
                    info['hostname'] = hostname

    for host in ['Enterprise0', 'Enterprise1']:
        if host in observation:
            iface_list = observation[host].get('Interface', [])
            for iface in iface_list:
                if 'IP Address' in iface:
                    ip = str(iface['IP Address'])
                    if host not in red_knowledge:
                        red_knowledge[host] = {'ip': ip}
                    if 'Subnet' in iface:
                        red_knowledge[host]['subnet'] = str(iface['Subnet'])

    for ip, hostname in ip_to_hostname.items():
        if hostname in ['Enterprise0', 'Enterprise1'] and hostname not in red_knowledge:
            red_knowledge[hostname] = {'ip': ip, 'hostname': hostname}

    for host, info in red_knowledge.items():
        ip = info.get('ip')
        if ip in ip_to_hostname:
            info['hostname'] = ip_to_hostname[ip]

    for ip_key, val in observation.items():
        if ip_key.count('.') != 3:
            continue
        sessions = val.get('Sessions', [])
        for sess in sessions:
            if sess.get('Agent') != 'Red':
                continue
            hostname = ip_to_hostname.get(ip_key)
            for host, info in red_knowledge.items():
                if info.get('ip') == ip_key:
                    info['session'] = sess['ID']
                    if hostname:
                        info['hostname'] = hostname

    return red_knowledge
#################################################################################################################
def red_state_vector(obs, red_knowledge, prev_state, Action_Name):
    if prev_state is None:
        state = np.zeros(red_state_dim)
    else:
        state = prev_state.copy()

    # [0:4] user sessions via observation key (IP) lookup
    ip_to_red_idx = red_knowledge.get('_ip_to_red_idx', {})
    for key, val in obs.items():
        if not isinstance(val, dict):
            continue
        for sess in val.get("Sessions", []):
            if sess.get("Agent") == "Red":
                username = sess.get("Username", "").lower()
                # try observation key as IP first
                if key in ip_to_red_idx:
                    idx = ip_to_red_idx[key]
                    state[idx] = 2 if username in ("root", "system") else 1
                else:
                    # fall back to Interface IP
                    for iface in val.get("Interface", []):
                        ip_str = str(iface.get("IP Address", ""))
                        if ip_str in ip_to_red_idx:
                            idx = ip_to_red_idx[ip_str]
                            state[idx] = 2 if username in ("root", "system") else 1
                            break

    # [4:6] enterprise IP known (persistent)
    if red_knowledge.get('Enterprise0', {}).get('ip'):
        state[4] = 1
    if red_knowledge.get('Enterprise1', {}).get('ip'):
        state[5] = 1

    # [6:8] enterprise services discovered
    if Action_Name == 'DiscoverNetworkServices_Enterprise0':
        ip = red_knowledge.get('Enterprise0', {}).get('ip')
        if ip in obs and "Processes" in obs[ip]:
            state[6] = 1
    if Action_Name == 'DiscoverNetworkServices_Enterprise1':
        ip = red_knowledge.get('Enterprise1', {}).get('ip')
        if ip in obs and "Processes" in obs[ip]:
            state[7] = 1

    # [8:10] enterprise sessions
    for i, ent in enumerate(['Enterprise0', 'Enterprise1']):
        host_keys = []
        if ent in obs:
            host_keys.append(ent)
        ip = red_knowledge.get(ent, {}).get('ip')
        if ip and ip in obs:
            host_keys.append(ip)
        for key in host_keys:
            val = obs.get(key)
            if not isinstance(val, dict):
                continue
            for sess in val.get("Sessions", []):
                if sess.get("Agent") == "Red":
                    username = sess.get("Username", "").lower()
                    red_knowledge[ent]["session"] = sess["ID"]
                    state[8+i] = 2 if username in ("root", "system") else 1

    # [10:12] ransomware flags
    if Action_Name == 'Impact_Enterprise0' and obs.get('success') == TrinaryEnum.TRUE:
        state[10] = 1
    if Action_Name == 'Impact_Enterprise1' and obs.get('success') == TrinaryEnum.TRUE:
        state[11] = 1

    # reset on failed actions
    for i in range(4):
        user = f'User{i+1}'
        if Action_Name in [f'ExploitRemoteService_{user}', f'PrivilegeEscalate_{user}']:
            if obs.get('success') == TrinaryEnum.FALSE:
                state[i] = 0

    for i, ent in enumerate(['Enterprise0', 'Enterprise1']):
        if Action_Name in [f'ExploitRemoteService_{ent}',
                           f'PrivilegeEscalate_{ent}',
                           f'Impact_{ent}']:
            if obs.get('success') == TrinaryEnum.FALSE:
                state[8+i] = 0
                state[10+i] = 0

    return state
#################################################################################################################
def Red_Action_Gen(idx, red_knowledge, p_exploit):
    # Map a red action index to (name, CybORG action). Actions red cannot take yet (unknown IP, hostname,
    # or session) become Sleep. Impact fails, becoming Sleep, with the host's disruption probability p.
    Action_Name = Red_Action_Space[int(idx)]
    if Action_Name == 'Sleep':
        return 'Sleep', Sleep()
    kind, host = Action_Name.split('_', 1)
    info = red_knowledge.get(host, {})
    if kind in ('DiscoverNetworkServices', 'ExploitRemoteService') and host in red_knowledge:
        cls = DiscoverNetworkServices if kind == 'DiscoverNetworkServices' else ExploitRemoteService
        return Action_Name, cls(ip_address=ipaddress.IPv4Address(info['ip']), session=0, agent='Red')
    if kind == 'PrivilegeEscalate' and 'hostname' in info:
        return Action_Name, PrivilegeEscalate(hostname=info['hostname'], session=0, agent='Red')
    if kind == 'Impact' and 'session' in info:
        if np.random.random() < p_exploit[0 if host == 'Enterprise0' else 1]:
            return 'Sleep', Sleep()
        return Action_Name, Impact(hostname=host, session=info['session'], agent='Red')
    return 'Sleep', Sleep()
#################################################################################################################
def logical_red_policy(red_state, type_theta):
    user_sess     = red_state[0:4]
    ent_ip_known  = [red_state[4], red_state[5]]
    ent_serv_disc = [red_state[6], red_state[7]]
    ent_sess      = [red_state[8], red_state[9]]
    ent_imp       = [red_state[10], red_state[11]]
    weights       = np.array(type_theta[0:2], dtype=float)
    A_SLEEP        = 0
    A_DISC_NS_ENT  = [1, 2]
    A_EXPLOIT_USER = [3, 4, 5, 6]
    A_EXPLOIT_ENT  = [7, 8]
    A_PRIVESC_USER = [9, 10, 11, 12]
    A_PRIVESC_ENT  = [13, 14]
    A_IMPACT_ENT   = [15, 16]
    def user_foothold():
        users_at_user = [i for i in range(4) if user_sess[i] == 1]
        if users_at_user: return A_PRIVESC_USER[random.choice(users_at_user)]
        users_uncomp = [i for i in range(4) if user_sess[i] == 0]
        if users_uncomp: return A_EXPLOIT_USER[random.choice(users_uncomp)]
        users_at_root = [i for i in range(4) if user_sess[i] == 2]
        if users_at_root: return A_PRIVESC_USER[random.choice(users_at_root)]
        return A_SLEEP
    target_priority = [i for i in range(2) if weights[i] > 0 and not (ent_sess[i] == 2 and ent_imp[i] == 1)]
    target_weights  = [weights[i] for i in target_priority]
    if not target_priority:
        reimpact = [i for i in range(2) if weights[i] > 0 and ent_sess[i] == 2 and ent_imp[i] == 1]
        if reimpact:
            rw = np.array([weights[i] for i in reimpact]); rw = rw / rw.sum()
            return A_IMPACT_ENT[np.random.choice(reimpact, p=rw)]
        return A_SLEEP
    in_progress = [i for i in target_priority if ent_sess[i] >= 1 and ent_imp[i] == 0]
    chosen_set  = in_progress if in_progress else target_priority
    tw = np.array([weights[i] for i in chosen_set], dtype=float); tw = tw / tw.sum()
    chosen = np.random.choice(chosen_set, p=tw)
    if ent_sess[chosen] == 2: return A_IMPACT_ENT[chosen]
    if ent_sess[chosen] == 1: return A_PRIVESC_ENT[chosen]
    if ent_serv_disc[chosen] == 1: return A_EXPLOIT_ENT[chosen]
    if ent_ip_known[chosen] == 1: return A_DISC_NS_ENT[chosen]
    return user_foothold()

## Blue side: observations, state component, actions

In [ ]:
def extract_blue_host_info(observation):
    tracked_hosts = ['User1', 'User2', 'User3', 'User4', 'Enterprise0', 'Enterprise1']
    blue_knowledge = {}

    for host in tracked_hosts:
        if host not in observation:
            continue

        host_info = observation[host]
        ip, subnet, session = None, None, None

        for iface in host_info.get('Interface', []):
            if ip is None and 'IP Address' in iface:
                ip = str(iface['IP Address'])
            if subnet is None and 'Subnet' in iface:
                subnet = str(iface['Subnet'])

        for sess in host_info.get('Sessions', []):
            if sess.get('Agent') == 'Blue':
                session = sess['ID']
                break

        if ip and subnet and session is not None:
            blue_knowledge[host] = {'ip': ip,'subnet': subnet,'session': session}

    return blue_knowledge
#################################################################################################################
def extract_y(obs, action_name):
    y = np.full(real_state_dim, -1, dtype=np.int8)
    host_indices = {'Monitor_User1_2':     [(0, 'User1'), (1, 'User2')],
                    'Monitor_User3_4':     [(2, 'User3'), (3, 'User4')],
                    'Monitor_Enterprise0': [(4, 'Enterprise0')],
                    'Monitor_Enterprise1': [(5, 'Enterprise1')]}
    if action_name in host_indices:
        for idx, hostname in host_indices[action_name]:
            host_obs = obs.get(hostname, {})
            processes = host_obs.get('Processes', [])
            level = 0
            for proc in processes:
                username = proc.get('Username', '')
                if username in ['root', 'SYSTEM']:
                    level = 2
                    break
                if username in ['pi', 'user', 'NetworkService', 'vagrant', 'www-data']:
                    level = 1
            y[idx] = level
    elif action_name == 'Analyse_Enterprise0':
        y[6] = 1 if obs.get('Enterprise0', {}).get('ImpactDetected') is True else 0
    elif action_name == 'Analyse_Enterprise1':
        y[7] = 1 if obs.get('Enterprise1', {}).get('ImpactDetected') is True else 0
    return y
#################################################################################################################
def blue_state_vector(y, prev_state, Action_Name, t):
    if prev_state is None:
        state = np.zeros(blue_state_dim)
    else:
        state = prev_state.copy()

    host_indices = {'Monitor_User1_2':     [0, 1],
                    'Monitor_User3_4':     [2, 3],
                    'Monitor_Enterprise0': [4],
                    'Monitor_Enterprise1': [5]}

    if Action_Name in host_indices:
        for idx in host_indices[Action_Name]:
            if y[idx] != -1:
                state[idx] = y[idx]

    if Action_Name == 'Analyse_Enterprise0' and y[6] != -1:
        state[6] = y[6]
    if Action_Name == 'Analyse_Enterprise1' and y[7] != -1:
        state[7] = y[7]

    if Action_Name == 'Remove_User1':        state[0] = 0
    if Action_Name == 'Remove_User2':        state[1] = 0
    if Action_Name == 'Remove_User3':        state[2] = 0
    if Action_Name == 'Remove_User4':        state[3] = 0
    if Action_Name == 'Remove_Enterprise0':  state[4] = 0; state[9]  = 0
    if Action_Name == 'Remove_Enterprise1':  state[5] = 0; state[10] = 0
    if Action_Name == 'Restore_Enterprise0': state[6] = 0; state[9]  = 0
    if Action_Name == 'Restore_Enterprise1': state[7] = 0; state[10] = 0

    state[8]  = 1 if t >= 3 else 0
    state[9]  = 1 if Action_Name == 'Analyse_Enterprise0' else 0
    state[10] = 1 if Action_Name == 'Analyse_Enterprise1' else 0

    return state
#################################################################################################################
MONITOR_HOSTS = {'Monitor_User1_2': ['User1', 'User2'], 'Monitor_User3_4': ['User3', 'User4'],
                 'Monitor_Enterprise0': ['Enterprise0'], 'Monitor_Enterprise1': ['Enterprise1']}

def Blue_Action_Gen(idx):
    # Map a blue action index to (name, CybORG action, host or hosts it concerns)
    Action_Name = Blue_Action_Space[int(idx)]
    if Action_Name == 'Sleep':
        return Action_Name, Sleep(), 'User0'
    if Action_Name in MONITOR_HOSTS:
        return Action_Name, Monitor(session=0, agent='Blue'), MONITOR_HOSTS[Action_Name]
    kind, host = Action_Name.split('_', 1)
    cls = {'Remove': Remove, 'Restore': Restore, 'Analyse': Analyse}[kind]
    return Action_Name, cls(hostname=host, session=0, agent='Blue'), host

## True state and payoffs

In [ ]:
def System_Real_State(state):
    vector = np.zeros(real_state_dim, dtype=np.int32)

    for i, host in enumerate(['User1', 'User2', 'User3', 'User4',
                               'Enterprise0', 'Enterprise1']):
        session_ids = state.hosts[host].sessions.get('Red', [])
        for sid in session_ids:
            try:
                sess = state.sessions['Red'][sid]
                username = sess.username.lower()
                if username in ['root', 'system']:
                    vector[i] = 2
                    break
                else:
                    vector[i] = max(vector[i], 1)
            except KeyError:
                continue

    for i, host in enumerate(['Enterprise0', 'Enterprise1']):
        files = state.hosts[host].files
        for file in files:
            if file.name == 'ransom.txt' and file.path == '/root/':
                vector[6+i] = 1
                break

    return vector
#################################################################################################################
# Defender's shaped training payoff
def Reward(next_real_state, w, action_name, y, prev_blue_state):
    r  = 0.1  * (8 - next_real_state[0:4].sum())
    r += 0.2  * (6 - next_real_state[4:6].sum())
    r += 2.0  * np.dot(w, 1 - next_real_state[6:8])

    host_indices    = {'Monitor_User1_2':     [0, 1],
                       'Monitor_User3_4':     [2, 3],
                       'Monitor_Enterprise0': [4],
                       'Monitor_Enterprise1': [5]}
    analyse_indices = {'Analyse_Enterprise0': 6,
                       'Analyse_Enterprise1': 7}
    remove_indices  = {'Remove_User1': 0, 'Remove_User2': 1,
                       'Remove_User3': 2, 'Remove_User4': 3,
                       'Remove_Enterprise0': 4, 'Remove_Enterprise1': 5}
    restore_indices = {'Restore_Enterprise0': 6,
                       'Restore_Enterprise1': 7}

    if action_name in host_indices:
        for idx in host_indices[action_name]:
            if y[idx] != -1 and y[idx] != prev_blue_state[idx]:
                r += 1.0
    elif action_name in analyse_indices:
        idx = analyse_indices[action_name]
        if y[idx] != -1 and y[idx] != prev_blue_state[idx]:
            r += 1.5
    elif action_name in remove_indices:
        idx = remove_indices[action_name]
        if prev_blue_state[idx] == 0:
            r -= 0.5
    elif action_name in restore_indices:
        idx = restore_indices[action_name]
        if prev_blue_state[idx] == 0:
            r -= 0.5
    elif action_name == 'Sleep':
        r -= 0.3
    return float(r)
#################################################################################################################
# Defender's physical payoff, used for evaluation
def Reward2(next_real_state, w):
    r  = 0.05  * (8 - next_real_state[0:4].sum())
    r += 0.5  * np.dot(w, 2 - next_real_state[4:6])
    r += 2.0  * np.dot(w, 1 - next_real_state[6:8])
    return float(r)

## Episodes: context, reset with red's reconnaissance, steps, history rows

In [ ]:
def Sample_Context():
    # theta = (p, w, h): disruption probabilities, red's target weights, red's head start
    w = np.array([0.0, 0.0])
    while w.sum() <= 0.0:
        w = np.random.randint(0, 2, size=2).astype(np.float32)
    w = w / w.sum()
    p_exploit = np.ones(2, dtype=np.float32)
    while all(p_exploit[i] == 1.0 for i in range(2) if w[i] > 0):     # at least one targeted host can be impacted
        for i in range(2):
            p_exploit[i] = np.random.choice([0.0, 0.5, 1.0])
    h = int(np.random.randint(0, 6))
    Theta = np.concatenate([p_exploit, w, [h]]).astype(np.float32)
    Type_Theta = Theta[2:Len_Theta]                                     # (w, h), known to red
    return Theta, Type_Theta, w, p_exploit, h


def Reset_Episode():
    # Reset the simulator and run red's initial reconnaissance of the user subnet
    blue_obs = cyborg.reset(agent='Blue')
    blue_knowledge = extract_blue_host_info(blue_obs.observation)
    blue_state = blue_state_vector(blue_obs.observation, np.zeros(blue_state_dim), 'Sleep', 0)

    red_obs = cyborg.get_observation('Red')
    red_knowledge = red_knowledge_dic(red_obs, {})
    red_state = red_state_vector(red_obs, red_knowledge, np.zeros(red_state_dim), 'Sleep')
    subnet = ipaddress.IPv4Network(red_knowledge['User0']['subnet'])
    res_red = cyborg.step(agent='Red', action=DiscoverRemoteSystems(subnet=subnet, session=0, agent='Red'))
    red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
    red_state = red_state_vector(res_red.observation, red_knowledge, red_state, 'Sleep')
    for label in ['User1', 'User2', 'User3', 'User4']:
        ip = ipaddress.IPv4Address(red_knowledge[label]['ip'])
        res_red = cyborg.step(agent='Red', action=DiscoverNetworkServices(ip_address=ip, session=0, agent='Red'))
        red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
        red_state = red_state_vector(res_red.observation, red_knowledge, red_state, 'Sleep')

    user_labels = ['User1', 'User2', 'User3', 'User4']
    red_knowledge['_ip_to_red_idx'] = {red_knowledge[label]['ip']: idx for idx, label in enumerate(user_labels)
                                       if red_knowledge.get(label, {}).get('ip')}
    real_state = System_Real_State(cyborg.environment_controller.state)
    return blue_state, blue_knowledge, red_state, red_knowledge, real_state


def red_step(u, red_knowledge, red_state, p_exploit):
    # Red's move: returns its action name, updated knowledge, updated private state, and the true state
    action_name, cyborg_action = Red_Action_Gen(u, red_knowledge, p_exploit)
    res_red = cyborg.step(agent='Red', action=cyborg_action)
    red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
    red_state = red_state_vector(res_red.observation, red_knowledge, red_state, action_name)
    return action_name, red_knowledge, red_state, System_Real_State(cyborg.environment_controller.state)


def blue_step(a):
    # Blue's move: returns its action name, its observation y, and the true state after both moves
    action_name, cyborg_action, _ = Blue_Action_Gen(a)
    res = cyborg.step(agent='Blue', action=cyborg_action)
    y = extract_y(res.observation, action_name)
    return action_name, y, System_Real_State(cyborg.environment_controller.state)


def head_start(policy, h, red_knowledge, red_state, Type_Theta, p_exploit):
    # h red-only steps before the episode starts, played by policy(red_state, Type_Theta)
    real_state = System_Real_State(cyborg.environment_controller.state)
    for _ in range(h):
        _, red_knowledge, red_state, real_state = red_step(policy(red_state, Type_Theta), red_knowledge, red_state, p_exploit)
    return red_knowledge, red_state, real_state


def Context_Row(blue_state, a, next_blue_state):
    row = np.zeros(CTX_DIM, dtype=np.float32)
    row[0:blue_state_dim] = blue_state
    row[blue_state_dim + a] = 1
    row[blue_state_dim + n_a:] = next_blue_state
    return row

def Context_Mask(num_rows):
    mask = torch.zeros((1, max_steps), dtype=torch.bool, device=device)
    mask[0, :max(1, min(num_rows, max_steps))] = True
    return mask

## Stage-game solver and replay buffer

In [ ]:
# Exact minimax by linear programming. Row player maximizes, column player minimizes.
# Returns (value, row strategy p, column strategy q).
def MatrixValue(M):
    M = np.asarray(M, dtype=float)
    if M.ndim != 2:
        raise ValueError("M must be a 2D array.")
    m, n = M.shape

    if m == 0 or n == 0 or not np.isfinite(M).all():
        return float(np.nanmean(M)), np.ones(m)/m if m else np.array([]), np.ones(n)/n if n else np.array([])

    # normalization
    center = float(M.mean())
    A = M - center
    scale = float(np.max(np.abs(A)))
    if scale == 0.0:  
        return center, np.ones(m)/m, np.ones(n)/n
    A /= scale  

    c1 = np.zeros(m + 1); c1[0] = -1.0  
    A_ub1 = np.zeros((n + m, m + 1))
    b_ub1 = np.zeros(n + m)
    A_ub1[:n, 0] =  1.0
    A_ub1[:n, 1:] = -A.T
    A_ub1[n:, 1:] = -np.eye(m)
    A_eq1 = np.ones((1, m + 1)); A_eq1[0, 0] = 0.0
    b_eq1 = np.array([1.0])

    opt1 = linprog(c=c1, A_ub=A_ub1, b_ub=b_ub1, A_eq=A_eq1, b_eq=b_eq1,method="highs", options={"presolve": True})
    v_row = None; p = None
    if opt1.success and opt1.x is not None:
        v_row = float(opt1.x[0])
        p = opt1.x[1:]

    c2 = np.zeros(n + 1); c2[0] = 1.0 
    A_ub2 = np.zeros((m + n, n + 1))
    b_ub2 = np.zeros(m + n)
    A_ub2[:m, 0]  = -1.0
    A_ub2[:m, 1:] =  A
    A_ub2[m:, 1:] = -np.eye(n)
    A_eq2 = np.ones((1, n + 1)); A_eq2[0, 0] = 0.0
    b_eq2 = np.array([1.0])

    opt2 = linprog(c=c2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2,
                   method="highs", options={"presolve": True})
    v_col = None; q = None
    if opt2.success and opt2.x is not None:
        v_col = float(opt2.x[0])
        q = opt2.x[1:]

    if p is None and q is None:
        return center, np.ones(m)/m, np.ones(n)/n
    if p is None:
        p = np.ones(m)/m
        v_norm = v_col
    elif q is None:
        q = np.ones(n)/n
        v_norm = v_row
    else:
        v_norm = 0.5 * (v_row + v_col)

    p = np.clip(p, 0.0, 1.0); ps = p.sum(); p = p/ps if ps > 0 else np.ones(m)/m
    q = np.clip(q, 0.0, 1.0); qs = q.sum(); q = q/qs if qs > 0 else np.ones(n)/n

    value = center + scale * float(v_norm)
    return value, p, q
#################################################################################################################
class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, args):
        self.memory.append(args)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

## Networks

In [ ]:
# Centralized Q(z^S, m^2, state, z^Theta, theta_type, theta) over all joint actions, the defender's payoff
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_bs1    = nn.Linear(blue_state_dim, 128)
        self.enc_bs2    = nn.Linear(128, 128)
        self.enc_embd1  = nn.Linear(Embedding_Size, 128)
        self.enc_embd2  = nn.Linear(128, 2)
        self.enc_rs1    = nn.Linear(red_state_dim, 128)
        self.enc_rs2    = nn.Linear(128, 128)
        self.enc_ss1    = nn.Linear(real_state_dim, 128)
        self.enc_ss2    = nn.Linear(128, 128)
        self.enc_type1  = nn.Linear(3, 128)
        self.enc_type2  = nn.Linear(128, 2)
        self.enc_theta1 = nn.Linear(Len_Theta, 128)
        self.enc_theta2 = nn.Linear(128, 128)
        self.ln_fuse    = nn.LayerNorm(128 * 4 + 2 * 2)
        self.fuse1      = nn.Linear(128 * 4 + 2 * 2, 512)
        self.fuse2      = nn.Linear(512, 256)
        self.out        = nn.Linear(256, n_a * n_u)

    def forward(self, bs, rs, ss, emb, type_theta, theta):
        z_b = F.relu(self.enc_bs2(F.relu(self.enc_bs1(bs))))
        z_m = F.relu(self.enc_rs2(F.relu(self.enc_rs1(rs))))
        z_x = F.relu(self.enc_ss2(F.relu(self.enc_ss1(ss))))
        z_c = F.relu(self.enc_embd2(F.relu(self.enc_embd1(emb))))
        z_y = F.relu(self.enc_type2(F.relu(self.enc_type1(type_theta))))
        z_t = F.relu(self.enc_theta2(F.relu(self.enc_theta1(theta))))
        e   = torch.cat([z_b, z_m, z_x, z_c, z_y, z_t], dim=-1)
        e   = self.ln_fuse(e)
        e   = F.relu(self.fuse1(e))
        e   = F.relu(self.fuse2(e))
        return self.out(e)
#################################################################################################################
# Context component z^Theta: Transformer with learned positional embeddings, masked mean pooling, feedforward head
class TSTransformer(nn.Module):
    def __init__(self):
        super(TSTransformer, self).__init__()
        self.input_projection = nn.Linear(blue_state_dim*2 + n_a, 128)
        self.position_embeddings = nn.Embedding(max_steps, 128)
        encoder_layers = TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True, dropout=0.1)
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers=3, enable_nested_tensor=False)
        self.fc1 = nn.Linear(128, 128)
        self.norm1 = nn.LayerNorm(128)
        self.fc2 = nn.Linear(128, Embedding_Size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)  

    def forward(self, x, mask):
        x = self.input_projection(x)  
        batch_size, seq_length, _ = x.shape 
        position_ids = torch.arange(seq_length, device=x.device).unsqueeze(0).expand(batch_size, -1)  
        position_embeddings = self.position_embeddings(position_ids)  
        x = x + position_embeddings
        key_padding_mask = ~mask.bool()
        x = self.transformer_encoder(x, src_key_padding_mask=key_padding_mask)
        mask = mask.unsqueeze(-1) 
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)  
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)  
        return x
#################################################################################################################
# pi^1(a | z), with z = [z^S, z^Theta]
class Player1Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_zs1  = nn.Linear(blue_state_dim, 128)
        self.enc_zs2  = nn.Linear(128, 128)
        self.enc_emb1 = nn.Linear(Embedding_Size, 128)
        self.enc_emb2 = nn.Linear(128, 128)
        self.fuse1 = nn.Linear(256, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_a)

    def forward(self, z_s, Embeding):
        zb = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        zc = F.relu(self.enc_emb2(F.relu(self.enc_emb1(Embeding))))
        h  = torch.cat([zb, zc], dim=-1)
        h  = self.ln(F.relu(self.fuse1(h)))
        h  = F.relu(self.fuse2(h))
        return F.softmax(self.head(h), dim=-1)
#################################################################################################################
# beta(u | m^2, w): red, acting on its private state and its target weights
class AttackerActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_x1   = nn.Linear(red_state_dim, 128)
        self.enc_x2   = nn.Linear(128, 128)
        self.enc_ty1  = nn.Linear(2, 64)
        self.enc_ty2  = nn.Linear(64, 128)
        self.fuse1 = nn.Linear(256, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_u)

    def forward(self, x, type_theta):
        zx = F.relu(self.enc_x2(F.relu(self.enc_x1(x))))
        zt = F.relu(self.enc_ty2(F.relu(self.enc_ty1(type_theta[..., :2]))))
        h = torch.cat([zx, zt], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        logits = self.head(h)
        return F.softmax(logits, dim=-1)  
#################################################################################################################
def red_policy(net):
    # Sampling policy (red_state, Type_Theta) -> action index for a red actor
    def act(red_state, Type_Theta):
        with torch.no_grad():
            probs = net(torch.tensor(red_state, dtype=torch.float32, device=device).view(1, -1),
                        torch.tensor(Type_Theta[0:2], dtype=torch.float32, device=device).view(1, -1)).squeeze(0).cpu().numpy()
        return int(np.random.choice(n_u, p=probs / probs.sum()))
    return act

## Representation step: encoder objective, Eq. (3)
$\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$:
- $\mathcal L_r$: regression of the stage reward on the representation, through a reward head.
- $\mathcal L_p$: prediction of the next representation from $(z_k, a^1_k, y_{k+1})$ through a latent transition model and a prediction head, against gradient-free exponential-moving-average copies of the encoder and projection. `F.mse_loss` averages over the 128 projection dimensions, so it is the squared norm in Eq. (3) divided by 128; the constant is absorbed into $\lambda_p$.
- $\mathcal L_{\mathrm{SI}}$: Soft-InfoNCE on the $\ell_2$-normalized projection, $\mathcal L_i = -\sum_{j\ne i} w_{ij}\log p_{ij}$, with $p_{ij}$ a softmax of $g_i^\top g_j/T$ over $k \ne i$ and graded targets $w_{ij} = s_{ij}/\sum_{\ell\ne i}s_{i\ell}$, $s_{ij} = \exp(-\tilde d_\Theta(\theta_i,\theta_j)/\sigma_\Theta)$.

The computable proxy is $\tilde d_\Theta = d_w + d_p + d_h$:
- $d_w = 1 - \cos(w_i, w_j)$ between the target weights,
- $d_p = \tfrac12\lVert p_i - p_j\rVert_1$ between the disruption probabilities, and
- $d_h = \lvert h_i - h_j\rvert / 5$ between the head starts (`Context_Similarity`).

In [ ]:
def Context_Similarity(th):
    # s_ij = exp(-(d_w + d_p + d_h) / sigma_Theta)
    p, w, h = th[:, 0:2], th[:, 2:4], th[:, 4]
    wn = F.normalize(w, p=2, dim=1)
    d_w = 1.0 - wn @ wn.t()
    d_p = 0.5 * torch.cdist(p, p, p=1)
    d_h = torch.abs(h.unsqueeze(1) - h.unsqueeze(0)) / 5.0
    return torch.exp(-(d_w + d_p + d_h) / SIGMA_THETA)
#################################################################################################################
def ema_update(tau=0.995):
    # Gradient-free targets for the next-representation loss
    for po, pt in zip(Context_Model.parameters(), target_model.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for po, pt in zip(proj_online.parameters(), proj_target.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for bo, bt in zip(Context_Model.buffers(), target_model.buffers()):
        bt.data.copy_(bo.data)
    for bo, bt in zip(proj_online.buffers(), proj_target.buffers()):
        bt.data.copy_(bo.data)
#################################################################################################################
def Train_Encoder():
    if len(context_memory) < CONTEXT_BATCH:
        return 0.0
    batch = context_memory.sample(CONTEXT_BATCH)
    (ctx, msk, nctx, nmsk, a, y, r, th) = map(list, zip(*batch))
    ctx  = torch.cat(ctx,  dim=0).to(device)
    msk  = torch.cat(msk,  dim=0).to(device)
    nctx = torch.cat(nctx, dim=0).to(device)
    nmsk = torch.cat(nmsk, dim=0).to(device)
    a  = torch.stack(a).to(device)
    y  = torch.stack(y).to(device).float()
    r  = torch.stack(r).to(device).float()
    th = torch.stack(th).to(device).float()
    Bsz = ctx.shape[0]

    Context_Model.train(); trans_pred.train(); reward_head.train()
    proj_online.train();  pred_online.train()
    z = Context_Model(ctx, msk)

    # L_r: reward prediction
    loss_r = F.mse_loss(reward_head(z).squeeze(-1), r)

    # L_p: next-representation prediction against the EMA target encoder and projection
    z_hat = trans_pred(torch.cat([z, F.one_hot(a, num_classes=n_a).float(), y], dim=-1))
    h_hat = F.normalize(pred_online(proj_online(z_hat)), p=2, dim=1)
    with torch.no_grad():
        h_tgt = F.normalize(proj_target(target_model(nctx, nmsk)), p=2, dim=1)
    loss_p = F.mse_loss(h_hat, h_tgt)

    # L_SI: Soft-InfoNCE with graded targets from the context similarity
    g = F.normalize(proj_online(z), p=2, dim=1)
    eye = torch.eye(Bsz, device=device, dtype=torch.bool)
    logits = ((g @ g.t()) / SI_TEMPERATURE).masked_fill(eye, -1e9)
    s = Context_Similarity(th).masked_fill(eye, 0.0)
    w = s / s.sum(dim=1, keepdim=True).clamp(min=1e-8)
    loss_si = -(w * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

    loss = LAMBDA_R * loss_r + LAMBDA_P * loss_p + LAMBDA_THETA * loss_si

    encoder_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(Context_Model.parameters()) + list(trans_pred.parameters()) +
        list(reward_head.parameters())  + list(proj_online.parameters()) +
        list(pred_online.parameters()), max_norm=1.0)
    encoder_optimizer.step()
    ema_update()

    Context_Model.eval()
    return loss.item()

## Game step: critic regression and actor distillation

In [ ]:
def _game_batch(batch_size):
    # Sample transitions and recompute the context embeddings with the current encoder
    batch = exp_memory.sample(batch_size)
    (bs, nbs, ss, nss, rs, nrs, a, u, r, tt, th, ctx, msk, nctx, nmsk) = map(list, zip(*batch))
    S = lambda v: torch.stack(v).to(device)
    C = lambda v: torch.cat(v, dim=0).to(device)
    Context_Model.eval()
    with torch.no_grad():
        emb  = Context_Model(C(ctx), C(msk))
        nemb = Context_Model(C(nctx), C(nmsk))
    return dict(bs=S(bs), nbs=S(nbs), ss=S(ss), nss=S(nss), rs=S(rs), nrs=S(nrs), a=S(a), u=S(u),
                r=S(r), tt=S(tt), th=S(th), emb=emb, nemb=nemb)
#################################################################################################################
def Train_Critic():
    if len(exp_memory) < CRITIC_BATCH:
        return 0.0
    b = _game_batch(CRITIC_BATCH)
    Bsz = b['bs'].shape[0]

    q_flat   = critic(b['bs'], b['rs'], b['ss'], b['emb'], b['tt'], b['th'])
    q_values = q_flat.gather(1, (b['a'] * n_u + b['u']).unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = critic_target(b['nbs'], b['nrs'], b['nss'], b['nemb'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        V_next = torch.tensor([MatrixValue(next_q[i])[0] for i in range(Bsz)], dtype=torch.float32, device=device)
        target_values = b['r'] + GAMMA * V_next

    loss = F.smooth_l1_loss(q_values, target_values)
    critic_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0)
    critic_optimizer.step()

    with torch.no_grad():
        for tgt, src in zip(critic_target.parameters(), critic.parameters()):
            tgt.data.mul_(1.0 - TAU).add_(TAU * src.data)
    return loss.item()
#################################################################################################################
def Train_Player1():
    # Defender: cross-entropy toward the row (maximizing) equilibrium strategy of the critic's stage game
    if len(exp_memory) < PLAYER1_BATCH:
        return 0.0
    b = _game_batch(PLAYER1_BATCH)
    Bsz = b['bs'].shape[0]
    with torch.no_grad():
        Q = critic(b['bs'], b['rs'], b['ss'], b['emb'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_a = torch.tensor(np.stack([MatrixValue(Q[i])[1] for i in range(Bsz)]), dtype=torch.float32, device=device)
    pred_a = Player1(b['bs'], b['emb'])
    loss = -(tgt_a * torch.log(pred_a + 1e-8)).sum(dim=1).mean()
    player1_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player1.parameters(), max_norm=1.0)
    player1_optimizer.step()
    return loss.item()
#################################################################################################################
def Train_Player2():
    # Red: cross-entropy toward the column (minimizing) equilibrium strategy, acting on its private state
    if len(exp_memory) < PLAYER2_BATCH:
        return 0.0
    b = _game_batch(PLAYER2_BATCH)
    Bsz = b['bs'].shape[0]
    with torch.no_grad():
        Q = critic(b['bs'], b['rs'], b['ss'], b['emb'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_u = torch.tensor(np.stack([MatrixValue(Q[i])[2] for i in range(Bsz)]), dtype=torch.float32, device=device)
    pred_u = Player2(b['rs'], b['tt'])
    loss = -(tgt_u * torch.log(pred_u + 1e-8)).sum(dim=1).mean()
    player2_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player2.parameters(), max_norm=1.0)
    player2_optimizer.step()
    return loss.item()

## Baseline architectures and the best-response attacker (shared by baselines, best_response, and eval)

In [ ]:
OBS_DIM = blue_state_dim + real_state_dim + n_a    # baseline observation: [z_s, y clipped at 0, previous action]
RNN_H = 128
LATENT_DIM = 16


class RecurrentAC(nn.Module):
    # Recurrent actor-critic over the baseline observation, optionally conditioned on an extra input
    def __init__(self, extra_dim=0, rnn_h=RNN_H):
        super().__init__()
        self.extra_dim = extra_dim
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        dd = rnn_h + extra_dim
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_a))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, extra=None, h0=None):
        out, hT = self.gru(obs, h0)
        feat = out if extra is None else torch.cat([out, extra], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT
#################################################################################################################
class VariBADEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        self.mu = nn.Linear(rnn_h, latent_dim)
        self.logvar = nn.Linear(rnn_h, latent_dim)
    def forward(self, obs, h0=None):
        out, hT = self.gru(obs, h0)
        return self.mu(out), self.logvar(out), hT
#################################################################################################################
class BRAttacker(nn.Module):
    # Recurrent best-response red: GRU over [red's private state, previous action one-hot], conditioned on the full theta
    def __init__(self, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(red_state_dim + n_u, rnn_h, batch_first=True)
        self.theta_enc = nn.Sequential(nn.Linear(Len_Theta, 64), nn.ReLU())
        dd = rnn_h + 64
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_u))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, theta, h0=None):
        out, hT = self.gru(obs, h0)
        feat = torch.cat([out, self.theta_enc(theta)], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT

## Loaders and defender wrappers (shared by best_response and eval)

In [ ]:
def Load(net, path):
    net.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return net.eval()


def Load_Baselines(save_dir="compared"):
    # The four trained baselines from baselines.ipynb
    ck_ = lambda name: torch.load(os.path.join(save_dir, name + ".pth"), map_location=device, weights_only=True)
    out = {}
    for name in ('RPPO', 'DR_RPPO'):
        ck = ck_(name)
        pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
        out[name] = {'policy': pol}
    ck = ck_("VARIBAD"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(2 * L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = VariBADEncoder(L, ck['config']['RNN_H']).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['VARIBAD'] = {'policy': pol, 'encoder': enc, 'latent_dim': L}
    ck = ck_("MMD")
    pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['p1']); pol.eval()
    out['MMD'] = {'policy': pol}
    return out
#################################################################################################################
# Defender wrappers: reset(blue_state, Theta, Type_Theta) at the start of an episode, act_dist(red_state, real_state)
# for the action distribution, observe(a, blue_action_name, y, t) after each round. They use the globals
# Context_Model, critic, Player1, and BASELINES, which each evaluation notebook loads.
class PipelineDefender:
    # 'Trained P1': our defender on [z^S, z^Theta]. 'Critic P1': the Oracle, the critic's equilibrium row
    # with privileged access to the true state, red's private state, and the context.
    def __init__(self, name):
        self.name = name
    def reset(self, blue_state, Theta, Type_Theta):
        self.th = torch.tensor(Theta, dtype=torch.float32, device=device)
        self.tt = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        self.blue_state = blue_state.copy()
        self.Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
        self.t = 0
    def act_dist(self, red_state, real_state):
        ctx = torch.tensor(self.Context, dtype=torch.float32, device=device).view(1, max_steps, CTX_DIM)
        bs = torch.tensor(self.blue_state, dtype=torch.float32, device=device)
        with torch.no_grad():
            Emb = Context_Model(ctx, Context_Mask(self.t)).squeeze(0)
            if self.name == 'Trained P1':
                p = Player1(bs, Emb).cpu().numpy()
            else:
                Q = critic(bs, torch.tensor(red_state, dtype=torch.float32, device=device),
                           torch.tensor(real_state, dtype=torch.float32, device=device),
                           Emb, self.tt, self.th).cpu().numpy().reshape(n_a, n_u)
                _, p, _ = MatrixValue(Q)
        p = np.clip(np.asarray(p, dtype=np.float64), 0, None)
        return p / p.sum()
    def observe(self, a, blue_action_name, y, t):
        next_blue_state = blue_state_vector(y, self.blue_state, blue_action_name, t + 1)
        self.Context[self.t, :] = Context_Row(self.blue_state, a, next_blue_state)
        self.blue_state = next_blue_state
        self.t += 1


class RandomDefender:
    def reset(self, blue_state, Theta, Type_Theta):
        pass
    def act_dist(self, red_state, real_state):
        return np.full(n_a, 1.0 / n_a)
    def observe(self, a, blue_action_name, y, t):
        pass


class BaselineDefender:
    # A trained baseline acting on [z_s, y clipped at 0, a_prev], with VariBAD's belief as extra input
    def __init__(self, name):
        self.name = name
        self.m = BASELINES[name]
    def reset(self, blue_state, Theta, Type_Theta):
        self.h = None
        self.internal_blue_state = blue_state.astype(np.float32).copy()
        self.y_cur = np.zeros(real_state_dim, dtype=np.float32)
        self.a_prev = np.zeros(n_a, dtype=np.float32)
        if self.name == 'VARIBAD':
            self.enc_h = None
    def _extra(self, obs):
        if self.name == 'VARIBAD':
            with torch.no_grad():
                mu, logvar, self.enc_h = self.m['encoder'](
                    torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.enc_h)
            mu = mu.view(-1).cpu().numpy()
            std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
            return np.concatenate([mu, std])
        return None
    def act_dist(self, red_state, real_state):
        obs = np.concatenate([self.internal_blue_state, self.y_cur, self.a_prev]).astype(np.float32)
        ex_np = self._extra(obs)
        obs_t = torch.tensor(obs, device=device).view(1, 1, OBS_DIM)
        ex_t = None if ex_np is None else torch.tensor(np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, _, self.h = self.m['policy'](obs_t, ex_t, self.h)
        p = F.softmax(logits.view(-1), dim=-1).cpu().numpy().astype(np.float64)
        return p / p.sum()
    def observe(self, a, blue_action_name, y, t):
        a_onehot = np.zeros(n_a, dtype=np.float32); a_onehot[a] = 1.0
        self.internal_blue_state = blue_state_vector(y, self.internal_blue_state, blue_action_name, t + 1)
        self.y_cur = np.maximum(y, 0).astype(np.float32)
        self.a_prev = a_onehot


def get_defender(name):
    if name in ('Critic P1', 'Trained P1'):
        return PipelineDefender(name)
    if name == 'Random P1':
        return RandomDefender()
    return BaselineDefender(name)